# OR / Procedure Room Utilization — Exploratory Analysis

This notebook walks through the `or_utilization` package on the synthetic sample
dataset (`data/sample_schedule.csv`), from raw CSV to utilization metrics to
a "what-if" schedule optimization simulation.

**Context:** this project grew out of watching first-hand how much OR time gets
lost to late starts, slow turnarounds, and last-minute cancellations while
working in a hospital surgery department. None of the data here is real patient
or hospital data — it's fully synthetic, generated by `scripts/generate_sample_data.py`
to have realistic statistical properties (cancellation rates, turnaround
distributions, cascading delays) without exposing anything sensitive.

In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
import matplotlib.pyplot as plt

from or_utilization.loader import load_schedule
from or_utilization.metrics import compute_metrics
from or_utilization.simulate import simulate_schedule_optimization, compare_scenarios

pd.set_option("display.max_columns", None)
%matplotlib inline

## 1. Load the schedule data

In [ ]:
df = load_schedule("../data/sample_schedule.csv")
df.head()

## 2. Compute utilization metrics

In [ ]:
metrics = compute_metrics(df)
metrics.overall

In [ ]:
metrics.daily_room_summary.head(10)

## 3. Visualize utilization by room

In [ ]:
room_avg = metrics.daily_room_summary.groupby("room_id")["utilization_pct"].mean().sort_values()
ax = room_avg.plot(kind="barh", figsize=(8, 4), color="#2E86AB")
ax.axvline(75, color="gray", linestyle="--", label="75% benchmark")
ax.set_xlabel("Average Utilization (%)")
ax.set_title("Average Room Utilization by Room")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Look at delay, overrun, and turnaround patterns

In [ ]:
completed = metrics.case_level[metrics.case_level["is_completed"]]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(completed["start_delay_min"].dropna(), bins=30, color="#F18F01")
axes[0].set_title("Start Delay (min)")

axes[1].hist(metrics.turnaround_times["turnaround_min"].dropna(), bins=30, color="#3B8686")
axes[1].set_title("Turnaround Time (min)")
plt.tight_layout()
plt.show()

## 5. Which surgeons/blocks have the most room for improvement?

Surgeons with high cancellation rates or large overruns are often the biggest
levers for utilization improvement — not because of blame, but because their
blocks are where the most unpredictable time is being lost.

In [ ]:
metrics.surgeon_summary.sort_values("cancellation_rate_pct", ascending=False).head(10)

## 6. Simulate schedule-optimization scenarios

The simulator asks: *if we improved turnaround time, cut start delays, and
tightened overruns by some amount, how much additional capacity would that
free up — without extending operating hours?*

We compare three scenarios: conservative, moderate, and aggressive process
improvement.

In [ ]:
scenarios = [
    simulate_schedule_optimization(
        metrics.daily_room_summary,
        turnaround_reduction_min=5, start_delay_reduction_pct=15, overrun_reduction_pct=10,
        scenario_name="conservative"
    ),
    simulate_schedule_optimization(
        metrics.daily_room_summary,
        turnaround_reduction_min=10, start_delay_reduction_pct=30, overrun_reduction_pct=25,
        scenario_name="moderate"
    ),
    simulate_schedule_optimization(
        metrics.daily_room_summary,
        turnaround_reduction_min=15, start_delay_reduction_pct=50, overrun_reduction_pct=40,
        scenario_name="aggressive"
    ),
]

compare_scenarios(scenarios)[[
    "scenario_name", "total_reclaimed_minutes",
    "est_additional_cases_from_efficiency_only",
    "baseline_utilization_pct", "projected_utilization_pct"
]]

## 7. Takeaways

- Even the **moderate** scenario — a realistic ask (10-minute faster turnarounds,
  30% less start delay, 25% less overrun) — recovers a meaningful number of
  case-equivalents per month without adding a single operating hour.
- The gap between `est_additional_cases_from_efficiency_only` and
  `est_additional_cases_incl_existing_idle` shows how much *already-idle* block
  time exists on top of process-improvement gains — that's often the more
  actionable number for a block-time reallocation conversation.
- These are transparent, assumption-driven estimates (see `simulate.py`
  docstring), not a full discrete-event simulation — every number is traceable
  back to a stated input so it can be challenged and refined by OR leadership.